# Entrega 4 - Extração de Entidades

**Curso:** Processamento de Linguagem Natural  
**Aluno:** Gisele Fonseca
**Data:** 06 de fev 2026

---

### Contexto e objetivo

Você irá implementar o módulo de **extração de entidades** do chatbot de e-commerce:

1. Utilizar o spaCy para reconhecimento de entidades nomeadas em português
2. Criar expressões regulares para extrair padrões específicos (pedidos, valores, datas)
3. Combinar NER e regex para extração robusta de informações
4. Preparar funções de extração para integração no chatbot final

O que você implementar aqui **será reutilizado cumulativamente** nas próximas entregas e no Projeto Final. Portanto, trate cada função como parte de uma API estável: assinaturas, nomes e tipos esperados devem ser preservados.

---

## Instruções

1. Complete todos os exercícios marcados com `# === SEU CÓDIGO AQUI ===`
2. **Não modifique** os nomes das funções - eles serão usados no Projeto Final
3. Execute todas as células em ordem antes de enviar
4. **Testes ocultos** serão executados na correção automática

---

## Critérios de Avaliação

| Questão | Tema | Pontos |
|---------|------|--------|
| Q1 | Extração de entidades com spaCy | 12 |
| Q2 | Análise de distribuição de entidades | 10 |
| Q3 | Regex para números de pedido | 12 |
| Q4 | Regex para valores monetários | 12 |
| Q5 | Regex para datas | 12 |
| Q6 | Regex para e-mails | 10 |
| Q7 | Regex para telefones | 10 |
| Q8 | Integração: extrair todas as entidades | 22 |
| **Total** | | **100** |

---

## Conteúdo Avaliado

- **Aula 7:** NER e Extração de Informação
- **Aula 8:** Transformers e BERT (conceitos aplicados)

---

## Funções a Implementar

```python
# NER com spaCy
extrair_entidades_spacy(texto) -> list[tuple]

# Extração com Regex
extrair_numeros_pedido(texto) -> list[str]
extrair_valores_monetarios(texto) -> list[str]
extrair_datas(texto) -> list[str]
extrair_emails(texto) -> list[str]
extrair_telefones(texto) -> list[str]

# Extração Combinada
extrair_todas_entidades(texto) -> dict
```

Notas:
- A implementação é avaliada pelo **comportamento observável**: saídas, tipos e chaves retornadas.
- Este notebook impõe técnicas específicas por design, pois o resultado será consumido por etapas posteriores.


---

## Configuração do Ambiente

In [1]:
# ============================================================
# CÉLULA DE CONFIGURAÇÃO - NÃO MODIFICAR
# ============================================================

import subprocess
import sys

# Instalação do spaCy e modelo português
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spacy"])
subprocess.check_call([sys.executable, "-m", "spacy", "download", "pt_core_news_sm", "-q"])

import re
import spacy
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any
from datetime import datetime

# Carregar modelo spaCy para português
nlp = spacy.load("pt_core_news_sm")

# ============================================================
# SEED E CONSTANTES DE AVALIAÇÃO - NÃO MODIFICAR
# ============================================================

SEED_AVALIACAO = 42

# Padrões regex obrigatórios (use como base)
REGEX_PEDIDO = r'(?:PED|ORD)-\d{5,10}|\b\d{6,10}\b'
REGEX_VALOR = r'R\$\s*\d{1,3}(?:\.\d{3})*(?:,\d{2})?'
REGEX_DATA = r'\d{2}[/-]\d{2}[/-]\d{4}'
REGEX_EMAIL = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
REGEX_TELEFONE = r'(?:\(\d{2}\)\s*)?\d{4,5}-?\d{4}|0800-?\d{3}-?\d{4}'

# ============================================================
# TEXTOS DE VALIDAÇÃO - NÃO MODIFICAR
# ============================================================

TEXTO_VALIDACAO_Q1 = "A empresa Magazine Luiza anunciou promoções em São Paulo durante o mês de janeiro."

TEXTO_VALIDACAO_Q3 = "Meu pedido PED-987654 ainda não chegou. Também tenho o pedido ORD-12345 pendente."

TEXTO_VALIDACAO_Q4 = "O produto custava R$ 1.299,99 mas paguei R$ 999,00 com desconto."

TEXTO_VALIDACAO_Q5 = "Comprei em 15/01/2026 e a entrega está prevista para 20-01-2026."

TEXTO_VALIDACAO_Q6 = "Entre em contato: suporte@loja.com.br ou vendas@empresa.com"

TEXTO_VALIDACAO_Q7 = "Ligue para (11) 98765-4321 ou 0800-123-4567 para atendimento."

TEXTO_VALIDACAO_Q8 = "Pedido PED-789012 no valor de R$ 2.500,00 feito em 10/01/2026. Contato: usuario@email.com.br ou (11) 98765-4321"

# ============================================================
# TEXTOS PARA ANÁLISE - NÃO MODIFICAR
# ============================================================

TEXTOS_ANALISE = [
    "Comprei na Amazon Brasil um notebook Dell por R$ 3.499,00 em 05/01/2026. Pedido PED-123456.",
    "A Americanas oferece frete grátis para São Paulo. Meu pedido ORD-98765 chegou ontem.",
    "Entrei em contato com suporte@magazineluiza.com.br sobre o pedido PED-555123 de R$ 899,90.",
    "O Mercado Livre tem ótimos preços. Comprei em 12/01/2026, total de R$ 150,00.",
    "Liguei para (21) 3456-7890 da Casas Bahia sobre minha compra de R$ 2.100,00.",
    "Pedido 12345678 da Shopee está em trânsito desde 08-01-2026. Valor: R$ 75,50.",
    "A Netshoes enviou e-mail para cliente@gmail.com confirmando pedido PED-777888.",
    "Submarino: promoção de eletrônicos! Smartphone por R$ 1.899,00. SAC: 0800-777-7474.",
    "Dafiti entregou meu pedido ORD-333444 em 18/01/2026. Paguei R$ 299,99.",
    "Reclamo do pedido PED-999000 feito em 02/01/2026. Contato: joao.silva@email.com ou (11) 91234-5678."
]

print("Ambiente configurado com sucesso!")
print(f"   - spaCy versão: {spacy.__version__}")
print(f"   - Modelo carregado: pt_core_news_sm")
print(f"   - Seed de avaliação: {SEED_AVALIACAO}")

Ambiente configurado com sucesso!
   - spaCy versão: 3.8.11
   - Modelo carregado: pt_core_news_sm
   - Seed de avaliação: 42


---

## Parte 1: Reconhecimento de Entidades Nomeadas (NER) com spaCy

### Questão 1: Extração de Entidades com spaCy (12 pontos)

### Objetivo

Implementar uma função que utilize o modelo spaCy `pt_core_news_sm` para extrair entidades nomeadas de um texto em português.

### Entradas esperadas

- `texto` (str): Texto em português para análise

### Processamento obrigatório

1. Processar o texto com o modelo spaCy (`nlp`)
2. Iterar sobre as entidades encontradas (`doc.ents`)
3. Para cada entidade, extrair: texto (`ent.text`), rótulo (`ent.label_`), posição inicial (`ent.start_char`), posição final (`ent.end_char`)
4. Retornar lista de dicionários com as informações

### Artefatos que devem existir ao final

```python
def extrair_entidades_spacy(texto: str) -> List[Dict[str, Any]]:
    # Retorna lista de dicionários com chaves:
    # 'texto', 'tipo', 'inicio', 'fim'
```

### Restrições técnicas

- Usar o objeto `nlp` já carregado (não recarregar o modelo)
- Retornar lista vazia se não houver entidades
- Manter a ordem de aparição das entidades no texto

### Observações de continuidade

Esta função será usada no **Projeto Final** para identificar entidades como nomes de empresas, locais e organizações mencionadas pelo cliente.

In [2]:
def extrair_entidades_spacy(texto: str) -> List[Dict[str, Any]]:
    """
    Extrai entidades nomeadas usando spaCy.

    Returns:
        List[Dict[str, Any]]: Lista de dicionários com:
            - 'texto': string da entidade
            - 'tipo': rótulo da entidade (ex: PER, LOC, ORG)
            - 'inicio': índice inicial no texto
            - 'fim': índice final no texto
    """
    # === SEU CÓDIGO AQUI ===

    doc = nlp(texto)

    entidades = []

    for ent in doc.ents:
        entidades.append({
            "texto": ent.text,
            "tipo": ent.label_,
            "inicio": ent.start_char,
            "fim": ent.end_char
        })

    return entidades

In [3]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q1 - NÃO MODIFICAR
# ============================================================

print("Validando Q1: extrair_entidades_spacy")
print("=" * 50)

resultado_q1 = extrair_entidades_spacy(TEXTO_VALIDACAO_Q1)

assert resultado_q1 is not None, "Função retornou None"
assert isinstance(resultado_q1, list), "Deve retornar uma lista"

if len(resultado_q1) > 0:
    assert isinstance(resultado_q1[0], dict), "Elementos devem ser dicionários"
    assert 'texto' in resultado_q1[0], "Dicionário deve ter chave 'texto'"
    assert 'tipo' in resultado_q1[0], "Dicionário deve ter chave 'tipo'"
    assert 'inicio' in resultado_q1[0], "Dicionário deve ter chave 'inicio'"
    assert 'fim' in resultado_q1[0], "Dicionário deve ter chave 'fim'"

# Verificar se encontra entidades conhecidas
textos_entidades = [e['texto'] for e in resultado_q1]
print(f"Entidades encontradas: {textos_entidades}")

print("\nQ1 validada com sucesso!")

Validando Q1: extrair_entidades_spacy
Entidades encontradas: ['Magazine Luiza', 'São Paulo']

Q1 validada com sucesso!


### Questão 2: Análise de Distribuição de Entidades (10 pontos)

### Objetivo

Criar uma função que analise a distribuição de tipos de entidades em uma lista de textos.

### Entradas esperadas

- `textos` (List[str]): Lista de textos para análise

### Processamento obrigatório

1. Para cada texto, extrair entidades usando `extrair_entidades_spacy`
2. Contar a frequência de cada tipo de entidade
3. Retornar dicionário com contagens

### Artefatos que devem existir ao final

```python
def analisar_distribuicao_entidades(textos: List[str]) -> Dict[str, int]:
    # Retorna dicionário {tipo_entidade: contagem}
```

### Restrições técnicas

- Usar a função `extrair_entidades_spacy` implementada em Q1
- Retornar dicionário vazio se não houver entidades

### Observações de continuidade

Esta análise ajuda a entender quais tipos de entidades são mais comuns nos textos de e-commerce.

In [4]:
def analisar_distribuicao_entidades(textos: List[str]) -> Dict[str, int]:
    """
    Analisa a distribuição de tipos de entidades em uma lista de textos.

    Args:
        textos: Lista de textos para análise

    Returns:
        Dicionário com contagem por tipo de entidade
    """
    # === SEU CÓDIGO AQUI ===
    contador = Counter()

    for texto in textos:
        ents = extrair_entidades_spacy(texto)
        for e in ents:
            contador[e["tipo"]] += 1

    return dict(contador)

In [5]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q2 - NÃO MODIFICAR
# ============================================================

print("Validando Q2: analisar_distribuicao_entidades")
print("=" * 50)

resultado_q2 = analisar_distribuicao_entidades(TEXTOS_ANALISE)

assert resultado_q2 is not None, "Função retornou None"
assert isinstance(resultado_q2, dict), "Deve retornar um dicionário"

print(f"Distribuição de entidades:")
for tipo, contagem in sorted(resultado_q2.items(), key=lambda x: -x[1]):
    print(f"  {tipo}: {contagem}")

# Verificar que todas as contagens são inteiros positivos
for tipo, contagem in resultado_q2.items():
    assert isinstance(contagem, int), f"Contagem de {tipo} deve ser inteiro"
    assert contagem > 0, f"Contagem de {tipo} deve ser positiva"

print("\nQ2 validada com sucesso!")

Validando Q2: analisar_distribuicao_entidades
Distribuição de entidades:
  LOC: 9
  PER: 8
  MISC: 7
  ORG: 3

Q2 validada com sucesso!


---

## Parte 2: Extração com Expressões Regulares

### Questão 3: Extração de Números de Pedido (12 pontos)

### Objetivo

Implementar uma função que extraia números de pedido de textos usando expressões regulares.

### Entradas esperadas

- `texto` (str): Texto para análise

### Processamento obrigatório

1. Usar o padrão `REGEX_PEDIDO` fornecido
2. Encontrar todos os matches no texto
3. Retornar lista de números de pedido encontrados

### Padrões válidos

- `PED-XXXXX` a `PED-XXXXXXXXXX` (5 a 10 dígitos após PED-)
- `ORD-XXXXX` a `ORD-XXXXXXXXXX` (5 a 10 dígitos após ORD-)
- Sequência de 6 a 10 dígitos isolados

### Artefatos que devem existir ao final

```python
def extrair_numeros_pedido(texto: str) -> List[str]:
    # Retorna lista de números de pedido
```

### Restrições técnicas

- Usar `re.findall()` com o padrão fornecido
- Retornar lista vazia se não houver matches

### Observações de continuidade

No **Projeto Final**, esta função será usada para identificar pedidos mencionados pelo cliente e consultar status no sistema.

In [6]:
def extrair_numeros_pedido(texto: str) -> List[str]:
    """
    Extrai números de pedido de um texto.

    Args:
        texto: Texto para análise

    Returns:
        Lista de números de pedido encontrados
    """
    # === SEU CÓDIGO AQUI ===
    matches = re.findall(REGEX_PEDIDO, texto)
    return matches if matches else []

In [7]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q3 - NÃO MODIFICAR
# ============================================================

print("Validando Q3: extrair_numeros_pedido")
print("=" * 50)

resultado_q3 = extrair_numeros_pedido(TEXTO_VALIDACAO_Q3)

assert resultado_q3 is not None, "Função retornou None"
assert isinstance(resultado_q3, list), "Deve retornar uma lista"
assert len(resultado_q3) >= 2, f"Deveria encontrar pelo menos 2 pedidos, encontrou {len(resultado_q3)}"
assert "PED-987654" in resultado_q3, "Deveria encontrar PED-987654"
assert "ORD-12345" in resultado_q3, "Deveria encontrar ORD-12345"

print(f"Pedidos encontrados: {resultado_q3}")
print("\nQ3 validada com sucesso!")

Validando Q3: extrair_numeros_pedido
Pedidos encontrados: ['PED-987654', 'ORD-12345']

Q3 validada com sucesso!


### Questão 4: Extração de Valores Monetários (12 pontos)

### Objetivo

Implementar uma função que extraia valores monetários em Reais de textos.

### Entradas esperadas

- `texto` (str): Texto para análise

### Processamento obrigatório

1. Usar o padrão `REGEX_VALOR` fornecido
2. Encontrar todos os valores monetários
3. Retornar lista de valores encontrados (como strings)

### Padrões válidos

- `R$ X,XX` (valores simples)
- `R$ X.XXX,XX` (valores com milhares)
- `R$ X.XXX.XXX,XX` (valores com milhões)
- Espaço após R$ é opcional

### Artefatos que devem existir ao final

```python
def extrair_valores_monetarios(texto: str) -> List[str]:
    # Retorna lista de valores monetários
```

### Restrições técnicas

- Manter formato original (não converter para float)
- Retornar lista vazia se não houver matches

### Observações de continuidade

No **Projeto Final**, valores monetários serão extraídos para identificar o valor envolvido em reclamações ou devoluções.

In [8]:
def extrair_valores_monetarios(texto: str) -> List[str]:
    """
    Extrai valores monetários em Reais de um texto.

    Args:
        texto: Texto para análise

    Returns:
        Lista de valores monetários encontrados
    """
    # === SEU CÓDIGO AQUI ===
    matches = re.findall(REGEX_VALOR, texto)
    return matches if matches else []

In [9]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q4 - NÃO MODIFICAR
# ============================================================

print("Validando Q4: extrair_valores_monetarios")
print("=" * 50)

resultado_q4 = extrair_valores_monetarios(TEXTO_VALIDACAO_Q4)

assert resultado_q4 is not None, "Função retornou None"
assert isinstance(resultado_q4, list), "Deve retornar uma lista"
assert len(resultado_q4) >= 2, f"Deveria encontrar pelo menos 2 valores, encontrou {len(resultado_q4)}"

# Verificar se encontra valores específicos
valores_encontrados = ' '.join(resultado_q4)
assert '1.299,99' in valores_encontrados or '1299,99' in valores_encontrados, "Deveria encontrar R$ 1.299,99"
assert '999,00' in valores_encontrados, "Deveria encontrar R$ 999,00"

print(f"Valores encontrados: {resultado_q4}")
print("\nQ4 validada com sucesso!")

Validando Q4: extrair_valores_monetarios
Valores encontrados: ['R$ 1.299,99', 'R$ 999,00']

Q4 validada com sucesso!


### Questão 5: Extração de Datas (12 pontos)

### Objetivo

Implementar uma função que extraia datas de textos em formatos brasileiros.

### Entradas esperadas

- `texto` (str): Texto para análise

### Processamento obrigatório

1. Usar o padrão `REGEX_DATA` fornecido
2. Encontrar todas as datas no texto
3. Retornar lista de datas encontradas

### Padrões válidos

- `dd/mm/aaaa` (formato com barras)
- `dd-mm-aaaa` (formato com hífens)

### Artefatos que devem existir ao final

```python
def extrair_datas(texto: str) -> List[str]:
    # Retorna lista de datas
```

### Restrições técnicas

- Manter formato original encontrado
- Não validar se a data é válida (ex: 31/02/2026)
- Retornar lista vazia se não houver matches

### Observações de continuidade

No **Projeto Final**, datas serão extraídas para identificar quando compras ou problemas ocorreram.

In [10]:
def extrair_datas(texto: str) -> List[str]:
    """
    Extrai datas de um texto.

    Args:
        texto: Texto para análise

    Returns:
        Lista de datas encontradas
    """
    # === SEU CÓDIGO AQUI ===
    matches = re.findall(REGEX_DATA, texto)
    return matches if matches else []

In [11]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q5 - NÃO MODIFICAR
# ============================================================

print("Validando Q5: extrair_datas")
print("=" * 50)

resultado_q5 = extrair_datas(TEXTO_VALIDACAO_Q5)

assert resultado_q5 is not None, "Função retornou None"
assert isinstance(resultado_q5, list), "Deve retornar uma lista"
assert len(resultado_q5) >= 2, f"Deveria encontrar pelo menos 2 datas, encontrou {len(resultado_q5)}"
assert "15/01/2026" in resultado_q5, "Deveria encontrar 15/01/2026"
assert "20-01-2026" in resultado_q5, "Deveria encontrar 20-01-2026"

print(f"Datas encontradas: {resultado_q5}")
print("\nQ5 validada com sucesso!")

Validando Q5: extrair_datas
Datas encontradas: ['15/01/2026', '20-01-2026']

Q5 validada com sucesso!


### Questão 6: Extração de E-mails (10 pontos)

### Objetivo

Implementar uma função que extraia endereços de e-mail de textos.

### Entradas esperadas

- `texto` (str): Texto para análise

### Processamento obrigatório

1. Usar o padrão `REGEX_EMAIL` fornecido
2. Encontrar todos os e-mails no texto
3. Retornar lista de e-mails encontrados

### Artefatos que devem existir ao final

```python
def extrair_emails(texto: str) -> List[str]:
    # Retorna lista de e-mails
```

### Restrições técnicas

- Manter formato original (não normalizar para minúsculas)
- Retornar lista vazia se não houver matches

### Observações de continuidade

E-mails extraídos podem ser usados para contato ou verificação de conta do cliente.

In [12]:
def extrair_emails(texto: str) -> List[str]:
    """
    Extrai endereços de e-mail de um texto.

    Args:
        texto: Texto para análise

    Returns:
        Lista de e-mails encontrados
    """
    # === SEU CÓDIGO AQUI ===
    matches = re.findall(REGEX_EMAIL, texto)
    return matches if matches else []

In [13]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q6 - NÃO MODIFICAR
# ============================================================

print("Validando Q6: extrair_emails")
print("=" * 50)

resultado_q6 = extrair_emails(TEXTO_VALIDACAO_Q6)

assert resultado_q6 is not None, "Função retornou None"
assert isinstance(resultado_q6, list), "Deve retornar uma lista"
assert len(resultado_q6) >= 2, f"Deveria encontrar pelo menos 2 e-mails, encontrou {len(resultado_q6)}"
assert "suporte@loja.com.br" in resultado_q6, "Deveria encontrar suporte@loja.com.br"
assert "vendas@empresa.com" in resultado_q6, "Deveria encontrar vendas@empresa.com"

print(f"E-mails encontrados: {resultado_q6}")
print("\nQ6 validada com sucesso!")

Validando Q6: extrair_emails
E-mails encontrados: ['suporte@loja.com.br', 'vendas@empresa.com']

Q6 validada com sucesso!


### Questão 7: Extração de Telefones (10 pontos)

### Objetivo

Implementar uma função que extraia números de telefone de textos.

### Entradas esperadas

- `texto` (str): Texto para análise

### Processamento obrigatório

1. Usar o padrão `REGEX_TELEFONE` fornecido
2. Encontrar todos os telefones no texto
3. Retornar lista de telefones encontrados

### Padrões válidos

- `(XX) XXXXX-XXXX` (celular com DDD)
- `(XX) XXXX-XXXX` (fixo com DDD)
- `XXXXX-XXXX` (sem DDD)
- `0800-XXX-XXXX` (linha gratuita)

### Artefatos que devem existir ao final

```python
def extrair_telefones(texto: str) -> List[str]:
    # Retorna lista de telefones
```

### Restrições técnicas

- Manter formato original encontrado
- Retornar lista vazia se não houver matches

### Observações de continuidade

Telefones podem ser usados para callback ou verificação de identidade.

In [14]:
def extrair_telefones(texto: str) -> List[str]:
    """
    Extrai números de telefone de um texto.

    Args:
        texto: Texto para análise

    Returns:
        Lista de telefones encontrados
    """
    # === SEU CÓDIGO AQUI ===
    matches = re.findall(REGEX_TELEFONE, texto)
    return matches if matches else []

In [15]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q7 - NÃO MODIFICAR
# ============================================================

print("Validando Q7: extrair_telefones")
print("=" * 50)

resultado_q7 = extrair_telefones(TEXTO_VALIDACAO_Q7)

assert resultado_q7 is not None, "Função retornou None"
assert isinstance(resultado_q7, list), "Deve retornar uma lista"
assert len(resultado_q7) >= 2, f"Deveria encontrar pelo menos 2 telefones, encontrou {len(resultado_q7)}"

# Verificar se encontra os padrões
telefones_str = ' '.join(resultado_q7)
assert '98765-4321' in telefones_str or '987654321' in telefones_str, "Deveria encontrar (11) 98765-4321"
assert '0800' in telefones_str, "Deveria encontrar 0800-123-4567"

print(f"Telefones encontrados: {resultado_q7}")
print("\nQ7 validada com sucesso!")

Validando Q7: extrair_telefones
Telefones encontrados: ['(11) 98765-4321', '0800-123-4567']

Q7 validada com sucesso!


---

## Parte 3: Integração

### Questão 8: Extração Integrada de Todas as Entidades (22 pontos)

### Objetivo

Implementar uma função **integradora** que combine todas as técnicas de extração (spaCy + regex) em uma única interface.

### Entradas esperadas

- `texto` (str): Texto para análise completa

### Processamento obrigatório

1. Extrair entidades com spaCy (usar `extrair_entidades_spacy`)
2. Extrair números de pedido (usar `extrair_numeros_pedido`)
3. Extrair valores monetários (usar `extrair_valores_monetarios`)
4. Extrair datas (usar `extrair_datas`)
5. Extrair e-mails (usar `extrair_emails`)
6. Extrair telefones (usar `extrair_telefones`)
7. Consolidar tudo em um dicionário estruturado

### Artefatos que devem existir ao final

```python
def extrair_todas_entidades(texto: str) -> Dict[str, Any]:
    # Retorna dicionário com estrutura:
    # {
    #     'spacy': [...],      # lista de dicts do spaCy
    #     'pedidos': [...],    # lista de strings
    #     'valores': [...],    # lista de strings
    #     'datas': [...],      # lista de strings
    #     'emails': [...],     # lista de strings
    #     'telefones': [...]   # lista de strings
    # }
```

### Restrições técnicas

- **OBRIGATÓRIO**: Usar as funções implementadas nas questões anteriores
- Todas as chaves devem existir, mesmo que com listas vazias
- Não duplicar entidades entre categorias

### Observações de continuidade

Esta é a função **mais importante** para o **Projeto Final**. Ela será expandida para `extrair_e_validar_entidades()` que adiciona validação de formato e retorna estrutura com valores válidos/inválidos.

In [16]:
def extrair_todas_entidades(texto: str) -> Dict[str, Any]:
    """
    Extrai todas as entidades de um texto usando spaCy e regex.

    Args:
        texto: Texto para análise completa

    Returns:
        Dicionário com todas as entidades extraídas por categoria
    """
    # === SEU CÓDIGO AQUI ===
    entidades_spacy = extrair_entidades_spacy(texto)
    pedidos = extrair_numeros_pedido(texto)
    valores = extrair_valores_monetarios(texto)
    datas = extrair_datas(texto)
    emails = extrair_emails(texto)
    telefones = extrair_telefones(texto)

    return {
        "spacy": entidades_spacy if entidades_spacy else [],
        "pedidos": pedidos if pedidos else [],
        "valores": valores if valores else [],
        "datas": datas if datas else [],
        "emails": emails if emails else [],
        "telefones": telefones if telefones else []
    }

In [17]:
# ============================================================
# CÉLULA DE VALIDAÇÃO Q8 - NÃO MODIFICAR
# ============================================================

print("Validando Q8: extrair_todas_entidades")
print("=" * 50)

resultado_q8 = extrair_todas_entidades(TEXTO_VALIDACAO_Q8)

assert resultado_q8 is not None, "Função retornou None"
assert isinstance(resultado_q8, dict), "Deve retornar um dicionário"

# Verificar estrutura
chaves_obrigatorias = ['spacy', 'pedidos', 'valores', 'datas', 'emails', 'telefones']
for chave in chaves_obrigatorias:
    assert chave in resultado_q8, f"Falta a chave '{chave}'"
    assert isinstance(resultado_q8[chave], list), f"'{chave}' deve ser uma lista"

# Verificar conteúdo específico
assert "PED-789012" in resultado_q8['pedidos'], "Deveria encontrar PED-789012"
assert any('2.500' in v or '2500' in v for v in resultado_q8['valores']), "Deveria encontrar R$ 2.500,00"
assert "10/01/2026" in resultado_q8['datas'], "Deveria encontrar 10/01/2026"
assert "usuario@email.com.br" in resultado_q8['emails'], "Deveria encontrar usuario@email.com.br"
assert any('98765' in t for t in resultado_q8['telefones']), "Deveria encontrar (11) 98765-4321"

print("Resultado da extração integrada:")
for chave, valores in resultado_q8.items():
    print(f"  {chave}: {valores}")

print("\nQ8 validada com sucesso!")

Validando Q8: extrair_todas_entidades
Resultado da extração integrada:
  spacy: [{'texto': 'Pedido PED-789012', 'tipo': 'PER', 'inicio': 0, 'fim': 17}, {'texto': 'R$', 'tipo': 'MISC', 'inicio': 30, 'fim': 32}, {'texto': 'usuario@email.com.br', 'tipo': 'ORG', 'inicio': 72, 'fim': 92}]
  pedidos: ['PED-789012']
  valores: ['R$ 2.500,00']
  datas: ['10/01/2026']
  emails: ['usuario@email.com.br']
  telefones: ['(11) 98765-4321']

Q8 validada com sucesso!


---


## Demonstração: Extração de Entidades

Esta seção valida a integração mínima do sistema implementado.

In [18]:
# ============================================================
# DEMONSTRAÇÃO - Execute após implementar todas as funções
# ============================================================

print("***DEMONSTRAÇÃO: Sistema de Extração de Entidades***")
print("=" * 60)

# Texto de exemplo simulando mensagem de cliente
texto_demo = """
Olá, meu nome é João Silva e estou com problemas no pedido PED-456789.
Comprei um notebook Dell na loja de São Paulo em 05/01/2026 por R$ 4.599,00.
Já liguei para (11) 3456-7890 e mandei e-mail para suporte@loja.com.br
mas ninguém resolve! O produto deveria ter chegado em 10-01-2026.
Podem me ajudar? Meu contato é joao.silva@gmail.com ou (11) 98765-4321.
"""

print("\nTexto de entrada:")
print(texto_demo)

print("\nEntidades extraídas:")
print("-" * 40)

resultado_demo = extrair_todas_entidades(texto_demo)

print(f"\nEntidades spaCy:")
for ent in resultado_demo['spacy']:
    print(f"   - {ent['texto']} ({ent['tipo']})")

print(f"\nPedidos: {resultado_demo['pedidos']}")
print(f"Valores: {resultado_demo['valores']}")
print(f"Datas: {resultado_demo['datas']}")
print(f"E-mails: {resultado_demo['emails']}")
print(f"Telefones: {resultado_demo['telefones']}")

print("\n" + "=" * 60)
print("==> Sistema de extração funcionando corretamente!")

***DEMONSTRAÇÃO: Sistema de Extração de Entidades***

Texto de entrada:

Olá, meu nome é João Silva e estou com problemas no pedido PED-456789.
Comprei um notebook Dell na loja de São Paulo em 05/01/2026 por R$ 4.599,00.
Já liguei para (11) 3456-7890 e mandei e-mail para suporte@loja.com.br
mas ninguém resolve! O produto deveria ter chegado em 10-01-2026.
Podem me ajudar? Meu contato é joao.silva@gmail.com ou (11) 98765-4321.


Entidades extraídas:
----------------------------------------

Entidades spaCy:
   - Olá (MISC)
   - João Silva (PER)
   - PED-456789 (LOC)
   - Dell (ORG)
   - de São Paulo (LOC)
   - R$ (MISC)
   - suporte@loja.com.br (LOC)
   - joao.silva@gmail.com (ORG)

Pedidos: ['PED-456789']
Valores: ['R$ 4.599,00']
Datas: ['05/01/2026', '10-01-2026']
E-mails: ['suporte@loja.com.br', 'joao.silva@gmail.com']
Telefones: ['(11) 3456-7890', '(11) 98765-4321']

==> Sistema de extração funcionando corretamente!


---

## Resumo - Funções para o Projeto Final

As funções implementadas nesta entrega serão expandidas no Projeto Final:

| Entrega 4 | Projeto Final |
|-----------|---------------|
| `extrair_todas_entidades()` | `extrair_e_validar_entidades()` |

### Expansão prevista:

A função `extrair_e_validar_entidades()` adicionará:

1. **Validação de pedidos**: verificar se tem 6+ dígitos
2. **Validação de valores**: verificar se 0 < valor < 100.000
3. **Validação de datas**: verificar se ano está entre 2020-2030
4. **Estrutura expandida**: `{'tipo': {'valores': [...], 'validos': [...]}}`

---

## Checklist de Entrega

Antes de submeter, valide objetivamente:

- [ ] Todas as funções estão implementadas e executam sem exceções.
- [ ] O notebook executa do início ao fim, em ordem, sem intervenção manual.
- [ ] Todas as funções preservam:
  - assinatura
  - tipos de retorno
  - parâmetros obrigatórios do classificador
- [ ] A demonstração final executa e retorna saídas compatíveis com a especificação.
- [ ] As métricas de avaliação são calculadas corretamente e retornam valores válidos no intervalo [0, 1].

**Envie via Moodle até a data limite.**